In [6]:
import time
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.home() / "icidea_llm_ids"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

LABEL_MAP = {0: "NORMAL", 1: "DOS", 2: "FUZZY", 3: "GEAR", 4: "RPM"}

# Window parameters
WINDOW_SIZE = 14    # frames per window
STRIDE      = 5   # frames between window starts

SEED = 42

t0 = time.time()
df = pd.read_parquet(ARTIFACTS_DIR / "section5_cleaned_data.parquet")
print(f"✓ Loaded {len(df):,} rows in {time.time()-t0:.2f}s")
print(f"  Window size: {WINDOW_SIZE} frames")
print(f"  Stride:      {STRIDE} frames")
print(f"  Max possible windows per class: ~{(200_000 - WINDOW_SIZE) // STRIDE:,}")

✓ Loaded 1,000,000 rows in 0.16s
  Window size: 14 frames
  Stride:      5 frames
  Max possible windows per class: ~39,997


In [7]:
def generate_windows(class_df, window_size, stride, class_label):
    """
    Slide a fixed-size window over temporally-sorted CAN frames.
    
    Returns a list of dicts, each containing:
      - 'frames': list of dicts (one per frame in the window)
      - 'label':  majority-vote class label (int)
      - 'window_start_ts': timestamp of first frame
      - 'window_end_ts':   timestamp of last frame
    """
    rows = class_df.reset_index(drop=True)
    n = len(rows)
    windows = []

    for start in range(0, n - window_size + 1, stride):
        end = start + window_size
        chunk = rows.iloc[start:end]

        # Build frame list
        frames = []
        for _, row in chunk.iterrows():
            frames.append({
                "timestamp": float(row["timestamp"]),
                "can_id":    int(row["can_id"]),
                "dlc":       int(row["dlc"]),
                "data":      [int(row[f"data{i}"]) for i in range(8)],
            })

        # Majority-vote label (unanimous for DOS/GEAR/RPM; may vary for NORMAL/FUZZY)
        label_counts = Counter(int(row["label"]) for _, row in chunk.iterrows())
        majority_label = label_counts.most_common(1)[0][0]

        windows.append({
            "frames":           frames,
            "label":            majority_label,
            "window_start_ts":  frames[0]["timestamp"],
            "window_end_ts":    frames[-1]["timestamp"],
        })

    return windows

In [8]:
all_windows = []

for lbl, name in LABEL_MAP.items():
    t0 = time.time()
    class_df = df[df["label"] == lbl].copy()
    windows = generate_windows(class_df, WINDOW_SIZE, STRIDE, lbl)
    all_windows.extend(windows)
    print(f"  {name:7s}  {len(windows):,} windows  ({time.time()-t0:.1f}s)")

print(f"\nTotal windows: {len(all_windows):,}")

  NORMAL   39,998 windows  (16.9s)
  DOS      39,998 windows  (15.3s)
  FUZZY    39,998 windows  (15.8s)
  GEAR     39,998 windows  (15.6s)
  RPM      39,998 windows  (15.8s)

Total windows: 199,990


In [9]:
# Check a normal window
sample = all_windows[0]
print("Sample NORMAL window:")
print(f"  Label:  {LABEL_MAP[sample['label']]}")
print(f"  Frames: {len(sample['frames'])}")
print(f"  Time span: {sample['window_end_ts'] - sample['window_start_ts']:.4f}s")
print(f"  First frame: {sample['frames'][0]}")
print(f"  Last frame:  {sample['frames'][-1]}")

# Check a DOS window
dos_start = sum(1 for w in all_windows if w["label"] == 0)
sample_dos = all_windows[dos_start]
print(f"\nSample DOS window:")
print(f"  Label:  {LABEL_MAP[sample_dos['label']]}")
print(f"  Frames: {len(sample_dos['frames'])}")
print(f"  Unique CAN IDs in window: {set(f['can_id'] for f in sample_dos['frames'])}")
print(f"  Unique payloads: {set(tuple(f['data']) for f in sample_dos['frames'])}")

# Label distribution across all windows
from collections import Counter
label_dist = Counter(w["label"] for w in all_windows)
print(f"\nWindow label distribution:")
for lbl in sorted(label_dist):
    print(f"  {LABEL_MAP[lbl]:7s}: {label_dist[lbl]:,}")

Sample NORMAL window:
  Label:  NORMAL
  Frames: 14
  Time span: 0.0300s
  First frame: {'timestamp': 1479121434.850202, 'can_id': 848, 'dlc': 8, 'data': [5, 40, 132, 102, 109, 0, 0, 162]}
  Last frame:  {'timestamp': 1479121434.880214, 'can_id': 848, 'dlc': 8, 'data': [5, 40, 180, 102, 109, 0, 0, 146]}

Sample DOS window:
  Label:  DOS
  Frames: 14
  Unique CAN IDs in window: {0}
  Unique payloads: {(0, 0, 0, 0, 0, 0, 0, 0)}

Window label distribution:
  NORMAL : 39,998
  DOS    : 39,998
  FUZZY  : 39,998
  GEAR   : 39,998
  RPM    : 39,998


In [10]:
import json

# Serialize frames list as JSON string for parquet storage
records = []
for w in all_windows:
    records.append({
        "frames_json":      json.dumps(w["frames"]),
        "label":            w["label"],
        "window_start_ts":  w["window_start_ts"],
        "window_end_ts":    w["window_end_ts"],
    })

windows_df = pd.DataFrame(records)
windows_df["label"] = windows_df["label"].astype("uint8")

artifact_path = ARTIFACTS_DIR / "section6_windows.parquet"
windows_df.to_parquet(artifact_path, index=False)
size_mb = artifact_path.stat().st_size / 1e6

print(f"✓ Saved {len(windows_df):,} windows to {artifact_path}")
print(f"  Size: {size_mb:.1f} MB")
print(f"  Columns: {list(windows_df.columns)}")
print(f"  Label distribution:")
print(windows_df["label"].value_counts().sort_index().rename(LABEL_MAP).to_string())

✓ Saved 199,990 windows to /Users/deepakpatnaik/icidea_llm_ids/artifacts/section6_windows.parquet
  Size: 33.4 MB
  Columns: ['frames_json', 'label', 'window_start_ts', 'window_end_ts']
  Label distribution:
label
NORMAL    39998
DOS       39998
FUZZY     39998
GEAR      39998
RPM       39998
